In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML

In [ ]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [ ]:
from package_files.benefits_defns import *

In [ ]:
from package_files.logit_model import *

In [ ]:
# set font
plt.rcParams.update({'font.size': 14})

In [ ]:
usdf_salary = pd.read_parquet('../data/us_10m_nointernship_2018_2024_benefits.parquet.gzip')

In [ ]:
# rename usdf_salary
usdf = usdf_salary

In [ ]:
len(usdf)

In [ ]:
usdf.columns

In [ ]:
# usdf['HAS_SALARY_INFO'] = usdf.apply(lambda x: 1 if pd.notnull(x['SALARY']) else 0, axis=1)

In [ ]:
usdf['HAS_SALARY_INFO'] = usdf['SALARY'].notnull().astype(int)

In [ ]:
# usdf['HAS_SALARY_INFO'] = np.where(usdf['SALARY'].notnull(), 1, 0)

In [ ]:
# usdf.rename(columns = {'has_salary_info': 'HAS_SALARY_INFO'}, inplace=True)

In [ ]:
benefits4

In [ ]:
benefit_colors

In [ ]:
for benefit in benefits4:
    grouped = usdf.groupby(['YEAR','AI ROLE', benefit])['HAS_SALARY_INFO'].mean().reset_index()
    grouped['HAS_SALARY_INFO'] = pd.to_numeric(grouped['HAS_SALARY_INFO'], errors='coerce')
    print(grouped.info())
    grouped['YEAR'] = pd.to_numeric(grouped['YEAR'], errors = 'coerce')
    grouped['YEAR'].unique()
    grouped['AI ROLE'] = grouped['AI ROLE'].astype(str)
    grouped[benefit] = grouped[benefit].astype(str)
    grouped = grouped.sort_values('YEAR')
    print(grouped['YEAR'].unique())
    print(grouped.isnull().sum())
    palette = [benefit_colors[benefit], 'gray']
    display(grouped.head())
    
    plt.figure(figsize = (10,6))
    sns.lineplot(
        data = grouped,
        x = 'YEAR', 
        y = 'HAS_SALARY_INFO', 
        hue = 'AI ROLE',
        style = benefit,
        # markers=False,
        errorbar = None, 
        palette=palette,
        style_order = ['True', 'False'],
        hue_order=['True','False']
    )
    plt.xticks(grouped['YEAR'].unique())
    plt.xlim(grouped['YEAR'].min(), grouped['YEAR'].max())
    handles, labels = plt.gca().get_legend_handles_labels()
    new_handles = handles
    new_labels = ['AI Role', 'Yes','No', f'{benefits_labels_map[benefit]}','Yes','No']
    if benefit == benefits4[0]:
        plt.legend(handles = new_handles, labels = new_labels, loc='upper left')
    else: 
        plt.legend().remove()
    plt.xlabel(None)
    plt.ylabel('Share of Jobs with Wage Information')    
    plt.title(benefits_labels_map[benefit])
    fig_path = '../figures/salary/wage_info'
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    plt.savefig(fig_path + f'/wage_info_{benefit}.png')
    plt.show()

In [ ]:
# Define the number of rows and columns for subplots
nrows = 2
ncols = 3

# Set up the overall figure
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 12))
axes = axes.flatten()  # Flatten to easily index axes

for i, benefit in enumerate(benefits4):
    ax = axes[i]
    grouped = usdf.groupby(['YEAR','AI ROLE', benefit])['HAS_SALARY_INFO'].mean().reset_index()
    grouped['HAS_SALARY_INFO'] = pd.to_numeric(grouped['HAS_SALARY_INFO'], errors='coerce')
    # print(grouped.info())
    grouped['YEAR'] = pd.to_numeric(grouped['YEAR'], errors = 'coerce')
    grouped['YEAR'].unique()
    grouped['AI ROLE'] = grouped['AI ROLE'].astype(str)
    grouped[benefit] = grouped[benefit].astype(str)
    grouped = grouped.sort_values('YEAR')
    # print(grouped['YEAR'].unique())
    # print(grouped.isnull().sum())
    palette = [benefit_colors[benefit], 'gray']
    # display(grouped.head())
    
    sns.lineplot(
        data = grouped,
        x = 'YEAR', 
        y = 'HAS_SALARY_INFO', 
        hue = 'AI ROLE',
        style = benefit,
        # markers=False,
        errorbar = None, 
        palette=palette,
        style_order = ['True', 'False'],
        hue_order=['True','False'], 
        ax = ax
    )
    ax.set_xticks(grouped['YEAR'].unique())
    ax.set_xlabel(None)
    if i == 0 or i == 3:
        ax.set_ylabel('Share of Jobs with Wage Information')
    else:
        ax.set_ylabel(None)
    ax.set_title(benefits_labels_map[benefit])

    # Customize legend for each subplot
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
        new_labels = ['AI Role', 'Yes', 'No', f'{benefits_labels_map[benefit]}', 'Yes', 'No']
        ax.legend(handles=handles, labels=new_labels, loc='upper left')
    else:
        ax.legend().remove()
    
plt.tight_layout()
plt.savefig(fig_path + '/wage_info_all.png')
plt.show()


# Models

In [ ]:
models = []
model_data = usdf[usdf['YEAR'] == 2023]
for benefit in benefits4:
    model = run_logit_model(data = model_data, dependent = 'HAS_SALARY_INFO', predictor = 'AI ROLE', binary_vars = [benefit])
    models.append(model)
    model.summary()

In [ ]:
stargazer = Stargazer(models)
stargazer.render_latex()



In [ ]:
with open(f'../exports/tables/wage_info_2023.tex', 'w') as f:
    f.write(stargazer.render_latex())

In [ ]:
display(HTML(stargazer.render_html()))

In [ ]:
usdf[usdf['AI ROLE']==True]

## Only AI Roles

In [ ]:
models2 = []
model_data_ai = usdf[(usdf['YEAR'] == 2023) & ( usdf['AI ROLE'] == True)]
for benefit in benefits4:
    model = run_logit_model(data = model_data_ai, dependent = 'HAS_SALARY_INFO', predictor = benefit)
    models2.append(model)
    model.summary()

In [ ]:
stargazer = Stargazer(models2)
stargazer.render_latex()

In [ ]:
display(HTML(stargazer.render_html()))

In [ ]:
with open(f'../exports/tables/wage_info_2023_ai.tex', 'w') as f:
    f.write(stargazer.render_latex())